# Adaptive RAG — pipeline notebook

Use **`python main.py …`** from the repo root for the same flows in the terminal. This notebook mirrors:

1. **Final** — ensure artifacts (scripts 02–05) then PubMedQA shootout.
2. Optional **probe** and **text baselines** (separate cells).

Open from repo root or from `notebooks/`; the first code cell resolves `REPO` and adds `src/` to `sys.path`.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "06_adaptive_rag_shootout.py").is_file():
    REPO = REPO.parent
if not (REPO / "scripts" / "06_adaptive_rag_shootout.py").is_file():
    raise FileNotFoundError("Could not find repo root (expected scripts/06_adaptive_rag_shootout.py).")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
print("REPO:", REPO)

## 1. Final pipeline (build if needed + shootout)

Set `SKIP_BUILD = True` to require existing `data/` and `artifacts/` (same as `python main.py final --skip-build`).

Default path uses full MedHallu + config model. To try a **small public model** and/or cap MedHallu rows for a faster run, set the variables in the next cell. Gated Llama needs **`HF_TOKEN`**; TinyLlama does not.

In [ ]:
import joblib
import torch
from huggingface_hub import login

from adaptive_rag.config import (
    ARTIFACTS_DIR,
    DISTILBERT_ROUTER_DIR,
    LORA_ROUTER_DIR,
    PUBMED_FAISS_INDEX,
    PUBMED_FAISS_MAPPING,
)
from adaptive_rag.orchestrate import ensure_final_artifacts
from adaptive_rag.report_benchmark import run_shootout_report_cli

SKIP_BUILD = False
N_SAMPLES = 100
# Optional: small HF id for 02/03/06 (same base). None = default from config (typically Llama).
GENERATOR_MODEL = os.environ.get("ADAPTIVE_RAG_GENERATOR_MODEL")
# GENERATOR_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Optional: first N MedHallu rows only (None = full split for TA-style runs).
MEDHALLU_MAX_ROWS = None
# MEDHALLU_MAX_ROWS = 128

tok = os.environ.get("HF_TOKEN")
if tok:
    login(token=tok)

ensure_final_artifacts(
    REPO,
    skip_build=SKIP_BUILD,
    hf_token=tok,
    medhallu_model_id=GENERATOR_MODEL,
    medhallu_max_rows=MEDHALLU_MAX_ROWS,
)

xgb_path = ARTIFACTS_DIR / "xgb_router.joblib"
xgb_model = joblib.load(xgb_path) if xgb_path.is_file() else None
if xgb_model is None:
    print("Note: no XGBoost joblib; shootout will omit Adaptive_XGBoost unless you train step 05.")

distil_dir = DISTILBERT_ROUTER_DIR if DISTILBERT_ROUTER_DIR.exists() else None
if distil_dir is None:
    print("Note: DistilBERT dir missing; D_BERT skipped.")

if not torch.cuda.is_available():
    print("Warning: CUDA not available.")

summary, df = run_shootout_report_cli(
    faiss_index_path=PUBMED_FAISS_INDEX,
    faiss_mapping_path=PUBMED_FAISS_MAPPING,
    lora_adapter_dir=LORA_ROUTER_DIR,
    distilbert_dir=distil_dir,
    xgb_model=xgb_model,
    n_samples=N_SAMPLES,
    print_table=True,
    csv_path=None,
    generator_model_id=GENERATOR_MODEL,
)
df

## 2. (Optional) Internal-state probe

HaluEval hidden-state linear probe — same as `python main.py probe`.

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, str(REPO / "scripts" / "01_internal_state_probe.py"), "--tasks", "qa"],
    cwd=str(REPO),
    check=False,
)

## 3. (Optional) Text baselines (HaluEval)

Same as `python main.py text-baselines --task qa --method tfidf`.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(REPO / "scripts" / "00_text_baselines_haluval.py"),
        "--task",
        "qa",
        "--method",
        "tfidf",
    ],
    cwd=str(REPO),
    check=False,
)